In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- mane_transcript_filter_row ---
FIX_MANE_TRANSCRIPT_FILTER_ROW_GENE = "BRCA1"
FIX_MANE_TRANSCRIPT_FILTER_ROW_PRIORITIZED_TX_ACS = ["NM_007294.4"]

# --- mane_transcript_parse_ac ---

print("✅ Fixtures loaded")
import logging as _logging; logger = _logging.getLogger(__name__)
DF_TRANSCRIPT_PD = pd.DataFrame({"ac_no_version_as_int": [1,2], "tx_ac": ["NM_000059.4","NM_007294.4"], "ac_version": ["4","4"], "alt_ac": ["NC_000013.11","NC_000017.11"]})
DF_TRANSCRIPT_PL = pl.from_pandas(DF_TRANSCRIPT_PD)
df = DF_TRANSCRIPT_PD


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_mane_transcript_filter_row(gene, prioritized_tx_acs):
    if df.empty:
        logger.warning(f"Unable to get transcripts from gene {gene}")
        return None

    for tx_ac in prioritized_tx_acs:
        tmp_df = df.loc[df["tx_ac"] == tx_ac].sort_values("alt_ac", ascending=False)
        row = tmp_df.iloc[0]
    return None

def before_mane_transcript_parse_ac(copy_df=None):
    if copy_df is None:
        copy_df = pd.DataFrame({"alt_ac":["NC_000001.11"],"tx_ac":["NM_001.1"],"alt_aln_method":["splign"],"tx_start_i":[0],"tx_end_i":[100],"alt_start_i":[0],"alt_end_i":[100],"alt_strand":[1]})
    copy_df["ac_no_version_as_int"] = copy_df["tx_ac"].apply(
        lambda x: int(x.split(".")[0].split("NM_")[1])
    )
    copy_df["ac_version"] = copy_df["tx_ac"].apply(lambda x: x.split(".")[1])
    copy_df = copy_df.sort_values(
        ["ac_no_version_as_int", "ac_version"], ascending=[False, False]
    )
    return copy_df

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_mane_transcript_filter_row(gene, prioritized_tx_acs):

    if df.is_empty():
        logger.warning(f"Unable to get transcripts from gene {gene}")
        return None

    for tx_ac in prioritized_tx_acs:
        tmp_df = df.filter(pl.col("tx_ac") == tx_ac).sort("alt_ac", descending=True)
        row = tmp_df.row(0, named=True)
    return None

def gen_mane_transcript_parse_ac(copy_df=None):
    if copy_df is None:
        copy_df = pl.DataFrame({"alt_ac":["NC_000001.11"],"tx_ac":["NM_001.1"],"alt_aln_method":["splign"],"tx_start_i":[0],"tx_end_i":[100],"alt_start_i":[0],"alt_end_i":[100],"alt_strand":[1]})

    copy_df = copy_df.with_columns(
        pl.col("tx_ac")
        .str.split(".")
        .list.get(0)
        .str.split("NM_")
        .list.get(1)
        .cast(pl.Int64)
        .alias("ac_no_version_as_int"),
        pl.col("tx_ac")
        .str.split(".")
        .list.get(1)
        .alias("ac_version"),
    )
    copy_df = copy_df.sort(
        by=["ac_no_version_as_int", "ac_version"],
        descending=[True, True],
    )
    return copy_df

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: mane_transcript_filter_row ===

import sys

def _call_filter_row_with(frame, func, gene, prioritized):
    global df
    old_df = df
    try:
        df = frame
        return func(gene, prioritized)
    finally:
        df = old_df

def _capture_filter_row(frame, func, gene, prioritized):
    captured = {}
    target_code = func.__code__
    def _trace(call_frame, event, arg):
        if event == "return" and call_frame.f_code is target_code:
            captured.update(call_frame.f_locals)
        return _trace
    old_trace = sys.gettrace()
    sys.settrace(_trace)
    try:
        result = _call_filter_row_with(frame, func, gene, prioritized)
    finally:
        sys.settrace(old_trace)
    return result, captured

def _row_as_dict(row, columns):
    if isinstance(row, pd.Series):
        return row.to_dict()
    if isinstance(row, dict):
        return row
    if isinstance(row, tuple):
        return dict(zip(columns, row))
    return row

try:
    _r = _call_filter_row_with(DF_TRANSCRIPT_PL, gen_mane_transcript_filter_row, FIX_MANE_TRANSCRIPT_FILTER_ROW_GENE, FIX_MANE_TRANSCRIPT_FILTER_ROW_PRIORITIZED_TX_ACS)
    print("✅ L1 smoke gen_mane_transcript_filter_row: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_mane_transcript_filter_row: {type(_e).__name__}: {_e}")

try:
    _rb = _call_filter_row_with(DF_TRANSCRIPT_PD, before_mane_transcript_filter_row, FIX_MANE_TRANSCRIPT_FILTER_ROW_GENE, FIX_MANE_TRANSCRIPT_FILTER_ROW_PRIORITIZED_TX_ACS)
    print("✅ L1 smoke before_mane_transcript_filter_row: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_mane_transcript_filter_row: {type(_e).__name__}: {_e}")

try:
    _rb, _before_locals = _capture_filter_row(DF_TRANSCRIPT_PD, before_mane_transcript_filter_row, FIX_MANE_TRANSCRIPT_FILTER_ROW_GENE, FIX_MANE_TRANSCRIPT_FILTER_ROW_PRIORITIZED_TX_ACS)
    _rg, _gen_locals = _capture_filter_row(DF_TRANSCRIPT_PL, gen_mane_transcript_filter_row, FIX_MANE_TRANSCRIPT_FILTER_ROW_GENE, FIX_MANE_TRANSCRIPT_FILTER_ROW_PRIORITIZED_TX_ACS)
    _before_row = _row_as_dict(_before_locals.get("row"), list(DF_TRANSCRIPT_PD.columns))
    _gen_row = _row_as_dict(_gen_locals.get("row"), list(DF_TRANSCRIPT_PL.columns))
    if _rb is None and _rg is None and _before_row == _gen_row:
        print("✅ L2 equivalence mane_transcript_filter_row selected row: MATCH")
    else:
        print(f"❌ L2 equivalence mane_transcript_filter_row selected row: MISMATCH — before={_before_row}, gen={_gen_row}")
except Exception as _e:
    print(f"❌ L2 equivalence mane_transcript_filter_row: setup error — {type(_e).__name__}: {_e}")

try:
    _rb = _call_filter_row_with(DF_TRANSCRIPT_PD.head(0), before_mane_transcript_filter_row, "MISSING", FIX_MANE_TRANSCRIPT_FILTER_ROW_PRIORITIZED_TX_ACS)
    _rg = _call_filter_row_with(DF_TRANSCRIPT_PL.head(0), gen_mane_transcript_filter_row, "MISSING", FIX_MANE_TRANSCRIPT_FILTER_ROW_PRIORITIZED_TX_ACS)
    if _rb is None and _rg is None:
        print("✅ L3 edge mane_transcript_filter_row empty table: MATCH")
    else:
        print(f"❌ L3 edge mane_transcript_filter_row empty table: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L3 edge mane_transcript_filter_row empty table: {type(_e).__name__}: {_e}")

try:
    _before_error = _gen_error = None
    try:
        _call_filter_row_with(DF_TRANSCRIPT_PD, before_mane_transcript_filter_row, FIX_MANE_TRANSCRIPT_FILTER_ROW_GENE, ["NM_missing.1"])
    except Exception as _e:
        _before_error = _e
    try:
        _call_filter_row_with(DF_TRANSCRIPT_PL, gen_mane_transcript_filter_row, FIX_MANE_TRANSCRIPT_FILTER_ROW_GENE, ["NM_missing.1"])
    except Exception as _e:
        _gen_error = _e
    if isinstance(_before_error, IndexError) and _gen_error is not None and not isinstance(_gen_error, (SyntaxError, NameError)):
        print(f"✅ L3 edge mane_transcript_filter_row missing tx: both sides reject (before={type(_before_error).__name__}, gen={type(_gen_error).__name__})")
    else:
        print(f"❌ L3 edge mane_transcript_filter_row missing tx: MISMATCH — before_error={_before_error}, gen_error={_gen_error}")
except Exception as _e:
    print(f"❌ L3 edge mane_transcript_filter_row missing tx: {type(_e).__name__}: {_e}")
